# Part 2 — NumPy & Pandas Data Wrangling

**TrendPulse:** engagement scores, cleaning, grouping, and how long videos stay trending.

### Notebook discipline
This notebook uses clear headings. Always use **Runtime → Restart and run all**.
All files are read from this Part's own `data/` folder (no internet needed).

### Colab tip
Colab cannot see your laptop. Upload a folder named `data` with these 4 files:
`USvideos.csv`, `GBvideos.csv`, `INvideos.csv`, `categories.csv`.

## 0. Setup

**What happens here:** We import libraries, choose the data folder, and save our engagement weights.

In [ ]:
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

# --- simple settings ---
COUNTRIES = ["US", "GB", "IN"]

# weights for the engagement score (add up to 1.0)
W_VIEWS = 0.50
W_LIKES = 0.30
W_COMMENTS = 0.20

# find the data folder (works on laptop and Colab)
possible_folders = [
    Path("data"),
    Path("/content/data"),
    Path("/content/part2_numpy_pandas/data"),
]

DATA_DIR = None
for folder in possible_folders:
    us_file = folder / "USvideos.csv"
    if us_file.exists():
        DATA_DIR = folder
        break

if DATA_DIR is None:
    print("Could not find data/USvideos.csv")
    print("In Colab: create /content/data and upload the 4 CSV files, then re-run.")
    print("Current folder:", Path.cwd())
    print("Files here:", list(Path(".").iterdir()))
    raise FileNotFoundError("Upload the data folder first.")

print("Using data folder:", DATA_DIR)
print("CSV files:", [p.name for p in DATA_DIR.glob("*.csv")])
print("TrendPulse Part 2 started at", datetime.now())
print("Weights -> views:", W_VIEWS, "likes:", W_LIKES, "comments:", W_COMMENTS)

### Why these engagement weights?
- **Views = 0.50:** how many people saw the video (most important for trending)
- **Likes = 0.30:** positive reaction
- **Comments = 0.20:** discussion (useful, but often smaller and noisier)

We normalize each metric to 0–1 first, then apply these weights.

## 1. NumPy arrays

**What happens here:** Load the CSV files, make NumPy arrays for views/likes/dislikes/comments, print shape and dtype, then practice slicing.

In [ ]:
# load each country file into a list of DataFrames
frames_raw = []

for code in COUNTRIES:
    file_path = DATA_DIR / (code + "videos.csv")
    one_country = pd.read_csv(file_path)
    one_country["country"] = code
    frames_raw.append(one_country)
    print("Loaded", code, "rows:", len(one_country))

# put all countries into one table
raw = pd.concat(frames_raw, ignore_index=True)
print("Total rows:", len(raw))

# make NumPy arrays (float so missing values can be NaN)
views = raw["views"].to_numpy(dtype=float)
likes = raw["likes"].to_numpy(dtype=float)
dislikes = raw["dislikes"].to_numpy(dtype=float)
comments = raw["comment_count"].to_numpy(dtype=float)

print("views shape:", views.shape, "dtype:", views.dtype)
print("likes shape:", likes.shape, "dtype:", likes.dtype)
print("dislikes shape:", dislikes.shape, "dtype:", dislikes.dtype)
print("comments shape:", comments.shape, "dtype:", comments.dtype)

# --- indexing and slicing practice ---
# 10 largest view counts
views_no_nan = views[~np.isnan(views)]
sorted_views = np.sort(views_no_nan)
top_10_views = sorted_views[-10:]
print("10 largest views:", top_10_views[::-1])

# every 100th record
every_100th = views[::100]
print("Every 100th view (first 5):", every_100th[:5])

# a custom sub-array
middle_slice = views[100:110]
print("Custom slice views[100:110]:", middle_slice)

## 2. Engagement score (NumPy only, no Python loops)

**What happens here:** We compute ratios, normalize, make a score, and flag the top 10%. All with NumPy array math.

In [ ]:
# replace missing numbers with 0 for scoring
views_clean = np.nan_to_num(views, nan=0.0)
likes_clean = np.nan_to_num(likes, nan=0.0)
dislikes_clean = np.nan_to_num(dislikes, nan=0.0)
comments_clean = np.nan_to_num(comments, nan=0.0)

# (a) like ratio = likes / (likes + dislikes)
# if denominator is 0, put 0 instead (avoids divide-by-zero)
like_denom = likes_clean + dislikes_clean
like_ratio = np.where(like_denom > 0, likes_clean / like_denom, 0.0)

# comment rate = comments / views
comment_rate = np.where(views_clean > 0, comments_clean / views_clean, 0.0)

print("like_ratio first 5:", like_ratio[:5])
print("comment_rate first 5:", comment_rate[:5])


def min_max_normalize(values):
    """Scale numbers to the range 0 to 1."""
    smallest = np.min(values)
    largest = np.max(values)
    spread = largest - smallest
    if spread == 0:
        return np.zeros_like(values)
    return (values - smallest) / spread


# (b) min-max normalize
views_norm = min_max_normalize(views_clean)
likes_norm = min_max_normalize(likes_clean)
comments_norm = min_max_normalize(comments_clean)

# (c) weighted engagement score
engagement = (W_VIEWS * views_norm) + (W_LIKES * likes_norm) + (W_COMMENTS * comments_norm)
print("engagement first 5:", engagement[:5])
print("engagement min:", engagement.min(), "max:", engagement.max())

# (d) log views
# Raw views go from small numbers to many millions.
# That makes charts hard to read. log1p compresses the scale.
# log1p(x) means log(1 + x), so 0 views stay safe.
log_views = np.log1p(views_clean)
print("log_views first 5:", log_views[:5])

# top 10% by engagement (boolean mask)
cutoff = np.quantile(engagement, 0.90)
top10_mask = engagement >= cutoff
print("Top 10% cutoff:", cutoff)
print("How many in top 10%:", top10_mask.sum())

## 3. Reshape and broadcast

**What happens here:** Make a table of daily counts (days × countries). Then use reshape, flatten, and broadcasting — still no cell-by-cell Python loops.

In [ ]:
# count how many trending rows each day for each country
daily_counts = raw.groupby(["trending_date", "country"]).size()
daily_table = daily_counts.unstack(fill_value=0)

# make sure columns are in US, GB, IN order
daily_table = daily_table.reindex(columns=COUNTRIES, fill_value=0)

# convert to a NumPy array: shape = (number of days, number of countries)
day_country = daily_table.to_numpy(dtype=float)
print("day_country shape:", day_country.shape)
print("First 3 days:\n", day_country[:3])

# flatten = make it 1D
flat = day_country.flatten()
print("flatten length:", len(flat))

# reshape back to original shape
n_days = day_country.shape[0]
n_countries = day_country.shape[1]
back = flat.reshape(n_days, n_countries)
print("reshape matches original?", np.array_equal(back, day_country))

# broadcasting 1: each country's share of that day's total
day_total = day_country.sum(axis=1, keepdims=True)   # shape (days, 1)
country_share = np.where(day_total > 0, day_country / day_total, 0.0)
print("Share on first day (US, GB, IN):", country_share[0])

# broadcasting 2: each day compared to that country's average
country_avg = day_country.mean(axis=0, keepdims=True)  # shape (1, countries)
vs_avg = np.where(country_avg > 0, day_country / country_avg, 0.0)
print("Vs country average on first day:", vs_avg[0])

## 4. Loading and inspection (Pandas)

**What happens here:** Load again as DataFrames, show head/info/describe/shape, then fix dates and types.

In [ ]:
country_frames = []

for code in COUNTRIES:
    file_path = DATA_DIR / (code + "videos.csv")
    df = pd.read_csv(file_path)
    df["country"] = code
    country_frames.append(df)

# inspect each country
i = 0
while i < len(country_frames):
    df = country_frames[i]
    code = COUNTRIES[i]
    print("=" * 50)
    print("Country:", code)
    print("shape:", df.shape)
    print(df.head())
    print(df.info())
    print(df.describe())
    i = i + 1

# combine all countries
combined = pd.concat(country_frames, ignore_index=True)
print("=" * 50)
print("COMBINED shape:", combined.shape)
print(combined.head())
print(combined.info())
print(combined.describe())

In [ ]:
# trending_date looks like 17.14.11  meaning year=17, day=14, month=11
combined["trending_date"] = pd.to_datetime(
    combined["trending_date"],
    format="%y.%d.%m",
    errors="coerce",
)

# publish_time is a normal timestamp string
combined["publish_time"] = pd.to_datetime(
    combined["publish_time"],
    errors="coerce",
    utc=True,
)

# make number columns numeric
number_cols = ["views", "likes", "dislikes", "comment_count", "category_id"]
for col in number_cols:
    combined[col] = pd.to_numeric(combined[col], errors="coerce")

# fix True/False flag columns
flag_cols = ["comments_disabled", "ratings_disabled", "video_error_or_removed"]
for col in flag_cols:
    combined[col] = combined[col].astype("boolean")

print(combined.dtypes)
print("trending_date examples:", combined["trending_date"].head(3).tolist())
print("publish_time examples:", combined["publish_time"].head(3).tolist())

### Data-quality issues (and how we handle them)

1. **Missing likes / comment_count** → `fillna(0)`
2. **Missing description** → `fillna("")`
3. **Bad dates after parsing** → `dropna(subset=["trending_date", "publish_time"])`
4. **Duplicate rows** (same video_id + trending_date + country) → `drop_duplicates`, keep first
5. **Odd date format** for trending_date → parse with `%y.%d.%m`
6. **publish_time as text** → convert with `pd.to_datetime`
7. **Removed / error videos** → keep the rows (flag is still useful)

## 5. Filtering with boolean masks, loc, iloc, sorting

**What happens here:** Add a temporary engagement score, then answer the three filter questions with `&` / `|`, `loc`, `iloc`, and sorting.

Full missing-value cleaning comes in the next section (assignment order).


In [ ]:
# engagement score for filtering (treat missing likes/comments as 0 for now)
v = combined["views"].fillna(0).to_numpy(dtype=float)
l = combined["likes"].fillna(0).to_numpy(dtype=float)
cmt = combined["comment_count"].fillna(0).to_numpy(dtype=float)

combined["engagement_score"] = (
    W_VIEWS * min_max_normalize(v)
    + W_LIKES * min_max_normalize(l)
    + W_COMMENTS * min_max_normalize(cmt)
)

# (a) 1 million+ views AND more than 50,000 likes
mask_a = (combined["views"] >= 1000000) & (combined["likes"] > 50000)
answer_a = combined[mask_a]
print("(a) rows with 1M+ views and 50k+ likes:", len(answer_a))
print(answer_a.loc[:, ["title", "country", "views", "likes"]].head())

# (b) top 20 by engagement score
answer_b = combined.sort_values("engagement_score", ascending=False).head(20)
print("\n(b) Top 20 by engagement_score:")
print(answer_b.loc[:, ["title", "channel_title", "country", "views", "engagement_score"]])

# (c) Music (10) OR Gaming (20), and views above overall median
median_views = combined["views"].median()
mask_c = (
    ((combined["category_id"] == 10) | (combined["category_id"] == 20))
    & (combined["views"] > median_views)
)
answer_c = combined[mask_c]
print("\n(c) Music/Gaming above median views", median_views, ":", len(answer_c), "rows")
print(answer_c.loc[:, ["title", "category_id", "views", "country"]].head())

# loc and iloc examples
first_index = combined.index[0]
print("\nloc title:", combined.loc[first_index, "title"])
print("iloc first 3 rows, first 4 columns:")
print(combined.iloc[0:3, 0:4])


## 6. Missing values and duplicates

**What happens here:** After filtering, we count missing values, fill or drop them, then remove duplicates.


In [ ]:
n_rows = len(combined)
print("Total rows before missing/duplicate cleaning:", n_rows)

# count missing values in each column (isna)
missing_count = combined.isna().sum()

print("\nMissing values (count and percent):")
for col in combined.columns:
    count = int(missing_count[col])
    if count > 0:
        percent = (count / n_rows) * 100
        print(col, ":", count, "(", round(percent, 2), "%)")

# fill missing values (fillna)
combined["likes"] = combined["likes"].fillna(0)
combined["comment_count"] = combined["comment_count"].fillna(0)
combined["description"] = combined["description"].fillna("")

print("\nWhy fill likes/comments with 0?")
print("So we can still score the row instead of deleting useful video info.")
print("Why fill description with empty text?")
print("We do not need description for the numeric questions in this Part.")

# dropna: remove rows where dates failed to parse
rows_before_dropna = len(combined)
combined = combined.dropna(subset=["trending_date", "publish_time"])
rows_after_dropna = len(combined)
print("\nRows dropped by dropna (bad dates):", rows_before_dropna - rows_after_dropna)
print("Why dropna on dates? Without dates we cannot compute days_to_trend.")

# duplicates = same video on the same day in the same country
dup_flags = combined.duplicated(
    subset=["video_id", "trending_date", "country"],
    keep="first",
)
n_dups = int(dup_flags.sum())
print("\nDuplicate rows found:", n_dups)

combined = combined.drop_duplicates(
    subset=["video_id", "trending_date", "country"],
    keep="first",
)
print("Rows after removing duplicates:", len(combined))
print("Duplicate means: same video_id + trending_date + country.")

# refresh engagement score on the cleaned table
v = combined["views"].to_numpy(dtype=float)
l = combined["likes"].to_numpy(dtype=float)
cmt = combined["comment_count"].to_numpy(dtype=float)
combined["engagement_score"] = (
    W_VIEWS * min_max_normalize(v)
    + W_LIKES * min_max_normalize(l)
    + W_COMMENTS * min_max_normalize(cmt)
)
print("\nEngagement score refreshed after cleaning.")


## 7. Join category names

**What happens here:** Merge `categories.csv` onto the trending rows.

We use a **left** merge so every trending row stays. If a category id is missing from the lookup, the name becomes empty/NaN, but the row is not deleted.

In [ ]:
categories = pd.read_csv(DATA_DIR / "categories.csv")
print(categories.head())

rows_before = len(combined)
combined = combined.merge(categories, on="category_id", how="left")
rows_after = len(combined)

print("Rows before merge:", rows_before)
print("Rows after merge:", rows_after)
print("Rows lost:", rows_before - rows_after)
print(combined.loc[:, ["category_id", "category_name"]].head())

## 8. Aggregation

**What happens here:** groupby + agg, value_counts, and best publish weekday/hour.

In [ ]:
# by category
by_category = combined.groupby("category_name").agg(
    record_count=("video_id", "size"),
    avg_views=("views", "mean"),
    avg_likes=("likes", "mean"),
    avg_engagement=("engagement_score", "mean"),
)
by_category = by_category.sort_values("record_count", ascending=False)
print("By category:")
print(by_category)

# by country
by_country = combined.groupby("country").agg(
    record_count=("video_id", "size"),
    avg_views=("views", "mean"),
    avg_likes=("likes", "mean"),
    avg_engagement=("engagement_score", "mean"),
)
print("\nBy country:")
print(by_country)

# category distribution
counts = combined["category_name"].value_counts()
percents = combined["category_name"].value_counts(normalize=True) * 100
print("\nCategory counts:")
print(counts)
print("\nCategory percents:")
print(percents.round(2))

# publish weekday and hour
combined["publish_weekday"] = combined["publish_time"].dt.day_name()
combined["publish_hour"] = combined["publish_time"].dt.hour

weekday_views = combined.groupby("publish_weekday")["views"].mean()
weekday_views = weekday_views.sort_values(ascending=False)
print("\nAverage views by publish weekday:")
print(weekday_views)

hour_views = combined.groupby("publish_hour")["views"].mean()
hour_views = hour_views.sort_values(ascending=False)
print("\nTop publish hours by average views:")
print(hour_views.head())

## 9. Time to trend and trending lifespan

**What happens here:**
- `days_to_trend` = trending date minus publish date
- lifespan = how many different days a video stayed on the trending list (per country)

In [ ]:
# make publish day comparable to trending_date
publish_day = combined["publish_time"].dt.tz_convert(None).dt.normalize()
trend_day = combined["trending_date"]

combined["days_to_trend"] = (trend_day - publish_day).dt.days

print("days_to_trend distribution:")
print(combined["days_to_trend"].describe())

# lifespan: count unique trending dates for each video in each country
lifespan = combined.groupby(["country", "video_id", "title"])["trending_date"].nunique()
lifespan = lifespan.reset_index()
lifespan = lifespan.rename(columns={"trending_date": "trending_lifespan_days"})

longest = lifespan.sort_values("trending_lifespan_days", ascending=False).head(10)
print("\n10 longest-lasting videos:")
print(longest)

# average lifespan by category
video_category = combined.groupby(["country", "video_id"], as_index=False).agg(
    category_name=("category_name", "first")
)
lifespan_with_cat = lifespan.merge(video_category, on=["country", "video_id"], how="left")

avg_life = lifespan_with_cat.groupby("category_name")["trending_lifespan_days"].mean()
avg_life = avg_life.sort_values(ascending=False)
print("\nAverage lifespan by category:")
print(avg_life)

## 10. Combining data with concat and merge

**What happens here:** Stack country tables with `concat`, then merge a per-video summary with a per-channel summary.

In [ ]:
# split cleaned data back into per-country frames, then stack with concat
us_df = combined[combined["country"] == "US"].copy()
gb_df = combined[combined["country"] == "GB"].copy()
in_df = combined[combined["country"] == "IN"].copy()

stacked = pd.concat([us_df, gb_df, in_df], ignore_index=True)
print("Stacked shape after concat:", stacked.shape)

# per-video summary
per_video = stacked.groupby(["country", "video_id", "channel_title"], as_index=False).agg(
    video_views=("views", "max"),
    days_on_list=("trending_date", "nunique"),
)

# per-channel summary
per_channel = stacked.groupby(["country", "channel_title"], as_index=False).agg(
    n_trending_videos=("video_id", "nunique"),
    total_views=("views", "sum"),
)

# merge the two summaries
channel_video = per_video.merge(
    per_channel,
    on=["country", "channel_title"],
    how="left",
)

print("Sample of merged video + channel summary:")
cols_to_show = [
    "country",
    "channel_title",
    "video_id",
    "n_trending_videos",
    "total_views",
    "video_views",
]
print(channel_video.loc[:, cols_to_show].head(10))

print("\nPart 2 finished successfully.")

## 11. Short wrap-up

We used beginner-friendly NumPy and Pandas steps to score engagement (views 0.5, likes 0.3, comments 0.2), clean missing values and duplicates, join category names with a left merge, and measure how fast and how long videos trend.